# 01 - Inventario de archivos originales y renombre de columnas

Inventaría los CSV originales de beneficiarios FONASA (2018-2025), detecta la codificación y el separador de cada archivo y lleva los nombres de columnas de todos los años a un esquema común. Se guarda una copia renombrada de cada año en Parquet; los CSV originales no se modifican.

**Entrada:** `base-beneficiarios-data/` - 8 CSV anuales (`beneficiarios_fonasa_<año>.csv`), en latin-1 o utf-8, separados por `|` o `,`.

**Salida:** `base-beneficiarios-data-renombrada/` - 8 archivos Parquet (`beneficiarios_fonasa_<año>.parquet`), con todas las columnas como texto y nombres de columnas homologados.

**Pasos**
1. Paso 1 - Construir el inventario de archivos
2. Paso 2 - Detectar codificación y separador
3. Paso 3 - Comparar columnas entre años
4. Paso 4 - Construir los diccionarios de renombre
   - Paso 4.1 - Imprimir plantilla de nombres en minúscula
   - Paso 4.2 - Definir el diccionario a minúscula
   - Paso 4.3 - Revisar la presencia de columnas tras pasar a minúscula
   - Paso 4.4 - Definir el diccionario al esquema común
5. Paso 5 - Guardar una copia renombrada de cada año en Parquet

Flujo del proyecto (los tres notebooks):

1. **01 · Inventario:** qué archivos hay y cuánto pesan.
2. **01 · Formato:** detectar la codificación y el separador de cada archivo, sin suponerlos.
3. **01 · Columnas:** comparar qué columnas trae cada año y renombrarlas a un esquema común.
4. **02 · Perfil:** revisar valores, vacíos y cambios de categorías entre años.
5. **02 · Limpieza:** definir y aplicar las reglas de limpieza, año por año.
6. **02 · Validación:** comprobar los totales por año contra las cifras publicadas por FONASA.
7. **03 · Agregación:** juntar los 8 años en una tabla para Power BI.

## Paso 1 - Construir el inventario de archivos

Lista cada CSV de la carpeta de datos originales, extrae el año del nombre del archivo con una expresión regular y registra el tamaño en MB. La tabla `inventario` resultante guía todos los ciclos posteriores, de modo que los archivos se procesan siempre año por año.

Resultado esperado: 8 filas (2018-2025), con tamaños entre aproximadamente 1.489 MB (2021) y 2.730 MB (2020).

In [1]:
from pathlib import Path
import pandas as pd
import re

carpeta = Path(r"C:\Users\ijoaq\PythonArchivos\PythonPractica\fondo-nacional-de-salud\base-beneficiarios-data")

inventario = []

for archivo in sorted(carpeta.glob("*.csv")): #itera sobre los archivos existentes que terminan en .csv
    #extrae un año de 4 dígitos aunque cambie la longitud del nombre
    match = re.search(r"(19|20)\d{2}", archivo.stem) #stem ignora el file extension (csv, pdf, etc.)
    
    inventario.append(
        {
        "ruta": archivo,
        "carpeta": archivo.parent.name,
        "archivo": archivo.name,
        #group para obtener texto encontrado por regex
        "anio": int(match.group()) if match else None,
        "tamanio_mb": round(archivo.stat().st_size / 1e6, 1) 
        }
    )

inventario = pd.DataFrame(inventario)
inventario

,ruta,carpeta,archivo,anio,tamanio_mb
0,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2018.csv,2018,2528.8
1,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2019.csv,2019,2671.0
2,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2020.csv,2020,2730.4
3,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2021.csv,2021,1488.7
4,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2022.csv,2022,2412.9
5,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2023.csv,2023,2524.5
6,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2024.csv,2024,2579.2
7,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2025.csv,2025,2305.7


La base completa suma aproximadamente 19,2 GB distribuidos en 8 archivos anuales, con tamaños que van desde 1,5 GB hasta 2,7 GB. Esto justifica trabajar año por año durante la exploración inicial, ya que cargar todos los archivos simultáneamente puede exceder la memoria disponible en equipos convencionales.

El archivo de 2021 destaca por su tamaño de 1,49 GB, cerca de un 45% menor que los años inmediatamente anteriores y posteriores. Dado que no se espera una reducción equivalente en la cantidad de beneficiarios, es probable que existan diferencias en el número de columnas, formatos de almacenamiento o niveles de detalle.

A partir de 2022 los tamaños vuelven al rango habitual de 2,4-2,6 GB, mientras que 2025 presenta una reducción hasta 2,31 GB respecto a 2024 (2,58 GB). Esta variación podría indicar cambios en la estructura de los datos, compresión del contenido o modificaciones en las variables incluidas.

Los años 2018-2020 son los archivos más pesados del conjunto pese a corresponder a los primeros años disponibles. Esto sugiere que el tamaño del archivo no depende únicamente de la cantidad de beneficiarios registrados, sino también de la cantidad de atributos almacenados y del formato utilizado en cada período.

Estas diferencias de tamaño constituyen una primera señal de que la estructura de los datos podría haber evolucionado entre años, por lo que conviene validar posteriormente la consistencia de columnas, tipos de datos y definiciones de variables antes de realizar análisis longitudinales.

## Paso 2 - Detectar codificación y separador

`detectar_formato` lee las primeras 10.000 líneas de cada archivo como bytes, intenta decodificarlas como utf-8 y, si falla, usa latin-1; el separador es el carácter (`,`, `;` o `|`) que más se repite en el encabezado. Así se evita fijar a mano parámetros de lectura que cambian entre años.

Resultado esperado: 2018-2020 usan `|` con 17 columnas; 2021 usa `,` con 9 columnas; 2022-2025 usan `,` con 12 columnas. Todos los años son latin-1 excepto 2025 (utf-8).

In [2]:
def detectar_formato(ruta):
    with open(ruta, "rb") as documento:
        lineas = [documento.readline() for linea in range(10_000)] #10000
    muestra = b"".join(lineas)

    try:
        muestra.decode("utf-8")
        encoding = "utf-8"
    except UnicodeDecodeError:
        encoding = "latin-1"

    encabezado = lineas[0].decode(encoding)
    sep = max([",", ";", "|"], key=encabezado.count)

    return {"encoding": encoding, "sep": sep, "n_columnas": encabezado.count(sep) + 1}

formatos = pd.DataFrame([detectar_formato(ruta) for ruta in inventario["ruta"]])
inventario = pd.concat([inventario, formatos], axis=1)
inventario    


,ruta,carpeta,archivo,anio,tamanio_mb,encoding,sep,n_columnas
0,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2018.csv,2018,2528.8,latin-1,|,17
1,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2019.csv,2019,2671.0,latin-1,|,17
2,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2020.csv,2020,2730.4,latin-1,|,17
3,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2021.csv,2021,1488.7,latin-1,",",9
4,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2022.csv,2022,2412.9,latin-1,",",12
5,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2023.csv,2023,2524.5,latin-1,",",12
6,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2024.csv,2024,2579.2,latin-1,",",12
7,C:\Users\ijoaq\PythonArchivos\PythonPractica\f...,base-beneficiarios-data,beneficiarios_fonasa_2025.csv,2025,2305.7,utf-8,",",12


## Paso 3 - Comparar columnas entre años

Lee solo el encabezado de cada archivo (`nrows=0`) con los parámetros detectados y construye un crosstab de nombre de columna por año (1 = presente). Esto muestra qué variables cambiaron de nombre o desaparecieron entre períodos.

Resultado esperado: 27 nombres de columna distintos; solo `EDAD_TRAMO` y `TITULAR_CARGA` mantienen el mismo nombre en los 8 años, y varias variables aparecen con nombres distintos (por ejemplo `SEXO_COR` / `SEXO`, `TRAMO` / `TRAMO_FONASA`).

In [3]:
registros = []
#intertuple: recorre el inventario fila por fila
for fila in inventario.itertuples():
    #archivo, filas, sep, encoding -> columnas
    columnas = pd.read_csv(fila.ruta, nrows=0, sep=fila.sep, encoding=fila.encoding).columns
    for col in columnas:
        registros.append({"anio": fila.anio, "columna": col})

registros = pd.DataFrame(registros)
#cada columna de cada archivo con un indicador si se repite en otro año (otro archivo)
presencia = pd.crosstab(registros["columna"], registros["anio"])
presencia

anio,2018,2019,2020,2021,2022,2023,2024,2025
columna,,,,,,,,
BENEF_ENCIPTADO,1,1,1,0,0,0,0,0
CARACTERIZACIÓN,0,0,0,0,1,1,1,1
COD_COMUNA_PERNAT,1,1,1,0,0,0,0,0
COD_REGION_PERNAT,1,1,1,0,0,0,0,0
COMUNA_BENEFICIARIO,0,0,0,1,1,1,1,1
CÓDIGO_BENEFICIARIO,0,0,0,0,1,1,1,1
DENSIDAD,1,1,1,0,0,0,0,0
DESC_COMUNA_PERNAT,1,1,1,0,0,0,0,0
DESC_REGION_PERNAT,1,1,1,0,0,0,0,0


## Paso 4 - Construir los diccionarios de renombre

### Paso 4.1 - Imprimir plantilla de nombres en minúscula

Imprime cada nombre de columna original como un par `'NOMBRE': 'nombre'`. La salida sirve de plantilla para escribir a mano el diccionario de la celda siguiente.

In [4]:
for nombre_columna in presencia.index:
    print(f"'{nombre_columna}': '{nombre_columna.lower()}'")

'BENEF_ENCIPTADO': 'benef_enciptado'
'CARACTERIZACIÓN': 'caracterización'
'COD_COMUNA_PERNAT': 'cod_comuna_pernat'
'COD_REGION_PERNAT': 'cod_region_pernat'
'COMUNA_BENEFICIARIO': 'comuna_beneficiario'
'CÓDIGO_BENEFICIARIO': 'código_beneficiario'
'DENSIDAD': 'densidad'
'DESC_COMUNA_PERNAT': 'desc_comuna_pernat'
'DESC_REGION_PERNAT': 'desc_region_pernat'
'DESC_TIPO_ASEGURADO': 'desc_tipo_asegurado'
'DESC_TIPO_CARACTERIZACION': 'desc_tipo_caracterizacion'
'EDAD_TRAMO': 'edad_tramo'
'ID_BENEFICIARIO': 'id_beneficiario'
'INCRITO_EAPS': 'incrito_eaps'
'MES_INFORMACION': 'mes_informacion'
'MES_INFORMACIÓN': 'mes_información'
'NACIONALIDAD': 'nacionalidad'
'NACIONALIDAD_COR': 'nacionalidad_cor'
'REGIÓN_BENEFICIARIO': 'región_beneficiario'
'SERVICIO_SALUD': 'servicio_salud'
'SEXO': 'sexo'
'SEXO_COR': 'sexo_cor'
'TIPO_ASEGURADO': 'tipo_asegurado'
'TITULAR_CARGA': 'titular_carga'
'TRAMO': 'tramo'
'TRAMO_FONASA': 'tramo_fonasa'
'TRAMO_RENTA': 'tramo_renta'


### Paso 4.2 - Definir el diccionario a minúscula

Define `nombres_columnas_minuscula`, que asigna a cada uno de los 27 nombres de columna originales su versión en minúscula.

In [5]:
nombres_columnas_minuscula = {
    'BENEF_ENCIPTADO': 'benef_enciptado',
'CARACTERIZACIÓN': 'caracterización',
'COD_COMUNA_PERNAT': 'cod_comuna_pernat',
'COD_REGION_PERNAT': 'cod_region_pernat',
'COMUNA_BENEFICIARIO': 'comuna_beneficiario',
'CÓDIGO_BENEFICIARIO': 'código_beneficiario',
'DENSIDAD': 'densidad',
'DESC_COMUNA_PERNAT': 'desc_comuna_pernat',
'DESC_REGION_PERNAT': 'desc_region_pernat',
'DESC_TIPO_ASEGURADO': 'desc_tipo_asegurado',
'DESC_TIPO_CARACTERIZACION': 'desc_tipo_caracterizacion',
'EDAD_TRAMO': 'edad_tramo',
'ID_BENEFICIARIO': 'id_beneficiario',
'INCRITO_EAPS': 'incrito_eaps',
'MES_INFORMACION': 'mes_informacion',
'MES_INFORMACIÓN': 'mes_información',
'NACIONALIDAD': 'nacionalidad',
'NACIONALIDAD_COR': 'nacionalidad_cor',
'REGIÓN_BENEFICIARIO': 'región_beneficiario',
'SERVICIO_SALUD': 'servicio_salud',
'SEXO': 'sexo',
'SEXO_COR': 'sexo_cor',
'TIPO_ASEGURADO': 'tipo_asegurado',
'TITULAR_CARGA': 'titular_carga',
'TRAMO': 'tramo',
'TRAMO_FONASA': 'tramo_fonasa',
'TRAMO_RENTA': 'tramo_renta'
}

### Paso 4.3 - Revisar la presencia de columnas tras pasar a minúscula

Repite el crosstab de encabezados del Paso 3 después de aplicar el diccionario a minúscula. Pasar a minúscula no fusiona ninguna columna, lo que confirma que hace falta un mapeo explícito entre nombres equivalentes.

Resultado esperado: las mismas 27 filas y el mismo patrón de presencia del Paso 3, ahora en minúscula.

In [6]:
registros_minuscula = []

for fila in inventario.itertuples():
    columnas_minusculas = pd.read_csv(filepath_or_buffer=fila.ruta, nrows=0, sep=fila.sep,encoding=fila.encoding)
    columnas_minusculas = columnas_minusculas.rename(columns=nombres_columnas_minuscula)
    for col in columnas_minusculas:
        registros_minuscula.append({"anio": fila.anio, "columna": col})

registros_minuscula = pd.DataFrame(registros_minuscula)
presencia_minuscula = pd.crosstab(registros_minuscula["columna"], registros_minuscula["anio"])
presencia_minuscula

anio,2018,2019,2020,2021,2022,2023,2024,2025
columna,,,,,,,,
benef_enciptado,1,1,1,0,0,0,0,0
caracterización,0,0,0,0,1,1,1,1
cod_comuna_pernat,1,1,1,0,0,0,0,0
cod_region_pernat,1,1,1,0,0,0,0,0
comuna_beneficiario,0,0,0,1,1,1,1,1
código_beneficiario,0,0,0,0,1,1,1,1
densidad,1,1,1,0,0,0,0,0
desc_comuna_pernat,1,1,1,0,0,0,0,0
desc_region_pernat,1,1,1,0,0,0,0,0


### Paso 4.4 - Definir el diccionario al esquema común

Define `nombres_columnas_actualizados`, que lleva los nombres equivalentes en minúscula a un único nombre de destino (por ejemplo `sexo_cor` → `sexo`, `tramo` → `tramo_fonasa`, `desc_region_pernat` → `región_beneficiario`, `benef_enciptado` y `código_beneficiario` → `id_beneficiario`).

In [7]:
nombres_columnas_actualizados = {
    "tramo": "tramo_fonasa",
    "sexo_cor": "sexo",
    "mes_información": "mes_informacion",
    "nacionalidad_cor": "nacionalidad",
    "desc_comuna_pernat": "comuna_beneficiario",
    "desc_region_pernat": "región_beneficiario" ,
    "benef_enciptado": "id_beneficiario",
    "código_beneficiario": "id_beneficiario",
    "desc_tipo_caracterizacion": "caracterización",
    "desc_tipo_asegurado": "tipo_asegurado"
}

## Paso 5 - Guardar una copia renombrada de cada año en Parquet

Lee cada CSV completo con su codificación y separador detectados (`dtype=str`, para que ningún valor se reinterprete), aplica ambos diccionarios de renombre y escribe el resultado en `base-beneficiarios-data-renombrada/`. Cada DataFrame se elimina después de guardarlo para limitar el uso de memoria; los CSV originales no se modifican.

Resultado esperado: una línea "Procesando documento del año" por cada año de 2018 a 2025, seguida de "Listo".

In [8]:
import pyarrow
carpeta_salida = carpeta.parent / "base-beneficiarios-data-renombrada"
carpeta_salida.mkdir(exist_ok=True)

for fila in inventario.itertuples():
    print("Procesando documento del año: ", fila.anio)
    df = pd.read_csv(filepath_or_buffer=fila.ruta, sep=fila.sep, encoding=fila.encoding, dtype=str)
    df = df.rename(columns=nombres_columnas_minuscula).rename(columns=nombres_columnas_actualizados)
    df.to_parquet(carpeta_salida / f"beneficiarios_fonasa_{fila.anio}.parquet", index=False)
    del df

print("Listo")

Procesando documento del año:  2018


Procesando documento del año:  2019


Procesando documento del año:  2020


Procesando documento del año:  2021


Procesando documento del año:  2022


Procesando documento del año:  2023


Procesando documento del año:  2024


Procesando documento del año:  2025


Listo
